In [4]:
import pandas as pd
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# Suppress common warnings for a cleaner output
warnings.filterwarnings('ignore')

def log_and_print(message, file_handle):
    """Helper function to print to console and write to a file."""
    print(message)
    file_handle.write(message + '\n')

def run_full_error_analysis(base_dir="evaluation_results", output_dir="full_error_analysis_output"):
    """
    Performs a comprehensive analysis of stylistically different texts for each stance.
    """
    print(f"🚀 Starting Full Error Analysis. Outputs will be saved to '{output_dir}'")
    
    # --- 0. Setup Directories ---
    plots_dir = os.path.join(output_dir, "feature_plots")
    os.makedirs(plots_dir, exist_ok=True)
    report_path = os.path.join(output_dir, "analysis_report.txt")

    # --- 1. Load Data and Construct Datasets from Indices (Done once) ---
    print("\n--- Part 1: Loading and Preparing Data ---")
    try:
        # (File loading logic is unchanged)
        path_wtwt_data = os.path.join(base_dir, 'wtwt_test_processed.csv')
        path_wtwt_correct_idx = os.path.join(base_dir, 'wtwt_correctly_classified_indices.npy')
        path_wtwt_mis_idx = os.path.join(base_dir, 'wtwt_misclassified_indices.npy')

        path_except_data = os.path.join(base_dir, 'except_wtwt_test_processed_mapped_data.csv')
        path_except_correct_idx = os.path.join(base_dir, 'except_wtwt_correctly_classified_indices.npy')
        path_except_mis_idx = os.path.join(base_dir, 'except_wtwt_misclassified_indices.npy')

        df_wtwt = pd.read_csv(path_wtwt_data)
        df_except_wtwt = pd.read_csv(path_except_data)

        wtwt_correct_indices = np.load(path_wtwt_correct_idx)
        wtwt_mis_indices = np.load(path_wtwt_mis_idx)
        except_correct_indices = np.load(path_except_correct_idx)
        except_mis_indices = np.load(path_except_mis_idx)

    except FileNotFoundError as e:
        print(f"❌ FATAL ERROR: Required file not found: {e.filename}")
        return

    df_correct_all = pd.concat([df_wtwt.iloc[wtwt_correct_indices], df_except_wtwt.iloc[except_correct_indices]], ignore_index=True)
    df_mis_all = pd.concat([df_wtwt.iloc[wtwt_mis_indices], df_except_wtwt.iloc[except_mis_indices]], ignore_index=True)

    df_correct_all["label"] = "Correct"
    df_mis_all["label"] = "Misclassified"
    
    if df_mis_all.empty:
        print("❌ FATAL ERROR: No misclassified examples found.")
        return

    misclassified_stance_counts = df_mis_all['stance'].value_counts()
    df_correct_sampled = df_correct_all.groupby('stance').apply(
        lambda x: x.sample(n=min(len(x), misclassified_stance_counts.get(x.name, 0)), random_state=42)
    ).reset_index(drop=True)

    df_balanced = pd.concat([df_correct_sampled, df_mis_all], ignore_index=True)
    df_balanced["label_bin"] = df_balanced["label"].map({"Correct": 1, "Misclassified": 0})
    print(f"✅ Data prepared. Balanced dataset has {len(df_balanced)} samples.")

    # --- 2. Iterate and Analyze Each Stance ---
    stances = [("All_Stances", None), ("FAVOR", 1), ("AGAINST", 0), ("NONE", 2)]
    feature_cols = [c for c in df_balanced.columns if c not in ['target', 'text', 'stance', 'label', 'label_bin', 'dataset', 'topic', 'split', 'index'] and pd.api.types.is_numeric_dtype(df_balanced[c])]

    with open(report_path, "w") as report_file:
        for stance_name, stance_val in stances:
            log_and_print(f"\n\n{'='*70}\n{'='*20} Analyzing Stance: {stance_name} {'='*20}\n{'='*70}", report_file)
            
            # a) Filter data for the current stance
            if stance_val is not None:
                df_stance = df_balanced[df_balanced['stance'] == stance_val].copy()
            else:
                df_stance = df_balanced.copy()
            
            if df_stance.empty or len(df_stance['label_bin'].unique()) < 2:
                log_and_print(f"⏩ Skipping {stance_name} due to insufficient data for both classes.", report_file)
                continue

            log_and_print(f"Analyzing {len(df_stance)} samples for stance '{stance_name}'.", report_file)
            
            # b) Train Models to Identify Important Features for this stance
            log_and_print("\n--- Identifying Key Differentiating Features ---", report_file)
            X = df_stance[feature_cols]
            y = df_stance["label_bin"]

            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
            scaler = StandardScaler().fit(SimpleImputer(strategy='median').fit_transform(X_train))
            X_train_scaled = scaler.transform(SimpleImputer(strategy='median').fit_transform(X_train))

            lr_model = LogisticRegression(random_state=42, max_iter=1000).fit(X_train_scaled, y_train)
            lr_importance = pd.DataFrame({'feature': feature_cols, 'coefficient': lr_model.coef_[0]}).assign(importance=lambda df: np.abs(df['coefficient'])).sort_values(by='importance', ascending=False)
            
            xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42).fit(X_train_scaled, y_train)
            xgb_importance = pd.DataFrame({'feature': feature_cols, 'importance': xgb_model.feature_importances_}).sort_values(by='importance', ascending=False)

            top_lr_features = lr_importance.head(10)
            top_xgb_features = xgb_importance.head(10)
            top_features = pd.concat([top_lr_features['feature'], top_xgb_features['feature']]).unique()[:10]
            
            log_and_print("\nLogistic Regression Top Features (Coefficient indicates direction: + for Correct, - for Misclassified):", report_file)
            log_and_print(top_lr_features[['feature', 'coefficient']].to_string(index=False), report_file)
            log_and_print("\n---\nXGBoost Top Features:", report_file)
            log_and_print(top_xgb_features[['feature', 'importance']].to_string(index=False), report_file)

            # c) Run Statistical & Visual Analysis on Top Features for this stance
            log_and_print("\n--- Statistical Tests and Plots for Top Features ---", report_file)
            
            stance_plots_dir = os.path.join(plots_dir, stance_name)
            os.makedirs(stance_plots_dir, exist_ok=True)

            for feature in top_features:
                log_and_print(f"\n- Analyzing Feature: '{feature}'", report_file)
                
                correct_data = df_stance[df_stance['label'] == 'Correct'][feature].dropna()
                misclassified_data = df_stance[df_stance['label'] == 'Misclassified'][feature].dropna()
                
                if len(correct_data) < 2 or len(misclassified_data) < 2:
                    log_and_print("  ⏩ Skipping t-test due to insufficient data in one or both groups.", report_file)
                    continue

                t_stat, p_value = stats.ttest_ind(correct_data, misclassified_data)
                log_and_print(f"  📊 T-test Result: p-value = {p_value:.4f}", report_file)
                if p_value < 0.05:
                    direction = "higher" if correct_data.mean() > misclassified_data.mean() else "lower"
                    log_and_print(f"  💡 Statistically Significant: 'Correct' texts have a {direction} '{feature}' value.", report_file)
                
                # Create and save plots
                plt.figure(figsize=(8, 6))
                sns.barplot(x="label", y=feature, data=df_stance, palette="viridis")
                plt.title(f"Distribution of '{feature}' for {stance_name}\n(p-value: {p_value:.4f})", fontsize=14)
                plt.savefig(os.path.join(stance_plots_dir, f"boxplot_{feature}.png"))
                plt.close()
                
                plt.figure(figsize=(10, 6))
                sns.histplot(data=df_stance, x=feature, hue="label", kde=True, common_norm=False)
                plt.title(f"Distribution of '{feature}' for {stance_name}", fontsize=14)
                plt.savefig(os.path.join(stance_plots_dir, f"histogram_{feature}.png"))
                plt.close()

            log_and_print(f"\n✅ All plots for {stance_name} saved in '{stance_plots_dir}'.", report_file)

            # d) Export Qualitative Sample for this stance
            stance_misclassified = df_stance[df_stance['label'] == 'Misclassified']
            if not stance_misclassified.empty:
                qualitative_sample = stance_misclassified.sample(n=min(50, len(stance_misclassified)), random_state=42)
                export_path = os.path.join(output_dir, f"qualitative_sample_{stance_name}.csv")
                qualitative_sample[['text', 'stance']].to_csv(export_path, index=False)
                log_and_print(f"✅ Qualitative sample for {stance_name} saved to '{export_path}'.", report_file)
                
    

    print(f"\n\n🎉 Analysis Finished! A full report is available at '{report_path}'.")
    return df_balanced
if __name__ == '__main__':
    sns.set_theme(style="whitegrid")
    df_balanced = run_full_error_analysis()

🚀 Starting Full Error Analysis. Outputs will be saved to 'full_error_analysis_output'

--- Part 1: Loading and Preparing Data ---
✅ Data prepared. Balanced dataset has 1516 samples.


==================== Analyzing Stance: All_Stances ====================
Analyzing 1516 samples for stance 'All_Stances'.

--- Identifying Key Differentiating Features ---

Logistic Regression Top Features (Coefficient indicates direction: + for Correct, - for Misclassified):
            feature  coefficient
         word_count    -0.770570
      hedge_C_count     0.634636
   type_token_ratio    -0.553086
      hedge_I_count    -0.452987
            I_ratio     0.326928
     hapax_legomena     0.319493
         verb_ratio    -0.306185
punctuation_density     0.302159
    avg_word_length    -0.299789
      hedge_D_count     0.219786

---
XGBoost Top Features:
                   feature  importance
                   D_ratio    0.081461
    doubt_adjectives_count    0.058610
   certainty_adverbs_count    0.0